In [1]:
import tskit 
import pandas as pd
import numpy as np
from lets_plot import *
LetsPlot.setup_html()
import pickle
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sys

chromosome = sys.argv[1]

In [3]:

print(chromosome)
ts = tskit.load("/sietch_colab/jobste/Singer/NewRun_CleanInds/Chr" + str(chromosome) + "/Chr" + str(chromosome) + "_NonHybrid_99.trees")

meta = pd.read_csv("/sietch_colab/jobste/Data/TsinferMetaData_Vcf_Country_corrected.csv")

samples = pd.read_csv("/sietch_colab/jobste/Singer/NewRun_CleanInds/SingerSamples_haploid.txt", header = None, names = ["ID"])

# %%
# Add subpop as pop
sampleMeta = samples.merge(meta, how = "left", on = "ID")


complex_schema_pop = tskit.MetadataSchema({
    'codec': 'json',
    'additionalProperties': False,
    'properties': {'pop': {'description': 'Lineage',
                        'type': 'string'}},
    'required': ['pop'],
    'type': 'object',
})

tables = ts.dump_tables()
tables.populations.metadata_schema = complex_schema_pop

# Add data to population table
popdata_column = [{"pop": pop} for pop in set(sampleMeta.Pop)]
encoded_popdata_column = [
    tables.populations.metadata_schema.validate_and_encode_row(r) for r in popdata_column
]
md, md_offset = tskit.pack_bytes(encoded_popdata_column)
tables.populations.set_columns(metadata=md, metadata_offset=md_offset)

popDict = {y: x for x,y in enumerate(set(sampleMeta.Pop))}
ind_nodes = list([x.id for x in ts.nodes() if x.time == 0.0])
ind_pops = [popDict[sampleMeta.Pop.to_list()[x]] for x in ind_nodes]


# %%
indNodePop = tables.nodes.population
indNodePop[ind_nodes] = ind_pops
tables.nodes.population = indNodePop
ts_meta = tables.tree_sequence()

ind_nodes_by_ind = [[x] for x in ind_nodes]

indPairs = []
for ind1 in ind_nodes:
    for ind2 in ind_nodes[(ind1+1):]:
        indPairs.append((ind1, ind2))
        
indPairsDivergence = ts_meta.divergence(sample_sets = ind_nodes_by_ind, indexes = indPairs)
#indPairsGRM = ts_meta.genetic_relatedness(sample_sets = ind_nodes_by_ind, indexes = indPairs)

indPairsDivergenceDF = pd.DataFrame({"Divergence": indPairsDivergence, "Pair": indPairs})
#indPairsGRMDF = pd.DataFrame({"IBS": indPairsGRM, "Pair": indPairs})



indPairsDivergenceDF.loc[:, "Ind1"] = [x[0] for x in indPairsDivergenceDF.Pair]
indPairsDivergenceDF.loc[:, "Ind2"] = [x[1] for x in indPairsDivergenceDF.Pair]
indPairsDivergenceDF.loc[:, "IndName1"] = [sampleMeta.ID[x] for x in indPairsDivergenceDF.Ind1]
indPairsDivergenceDF.loc[:, "IndName2"] = [sampleMeta.ID[x] for x in indPairsDivergenceDF.Ind2]
indPairsDivergenceDF.loc[:, "Pop1"] = [ts_meta.population(ts_meta.node(x).population).metadata["pop"] for x in indPairsDivergenceDF.Ind1]
indPairsDivergenceDF.loc[:, "Pop2"] = [ts_meta.population(ts_meta.node(x).population).metadata["pop"] for x in indPairsDivergenceDF.Ind2]

indPairsDivergenceDF.to_csv("IndPairsDivergence_Chr" + str(chromosome) + ".csv", index = None)

14


ValueError: Can only compare identically-labeled Series objects